# Step 4: T-cell gene-signature scoring and spatial neighbor composition

Scores T cells against three previously published gene-signature lists
(Cytotoxicity, Exhaustion, TIL dysfunction) using `sc.tl.score_genes`, and
quantifies the spatial neighborhood composition of each T-cell subtype.
Scores are compared across T-cell subtypes (per-cell boxplots), and the
per-patient mean score is contrasted between DX and PT timepoints with
paired t-tests and Wilcoxon tests; p-values are Benjamini-Hochberg adjusted
across the three signatures.

Inputs in `data/processed/`:

- `ext_fig3d_i_marker_lists.csv` (Cytotoxicity, Exhaustion, TIL dysfunction lists)

CSV outputs in `data/processed/`:

- `ext_fig3{d,e,f}_boxplot.csv` (per-cell score by T-cell subtype, one file per signature)
- `ext_fig3{g,h,i}_pointplot_data.csv` and `ext_fig3{g,h,i}_pointplot_stats.csv` (per-patient DX vs PT mean score and paired tests)
- `ext_fig3ghi_fdr.csv` (BH-adjusted p-values across signatures)
- `ext_fig4a.csv` (top-10 main-cell-type neighbors per T-cell subtype, per sample and consensus)
- `fig4a.csv` (consensus rows of the above)


## Setup and imports

In [ ]:
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
import numpy as np
import os
from scipy.stats import wilcoxon, ttest_rel
from matplotlib.patches import Patch
from statsmodels.stats.multitest import fdrcorrection

sc.settings.verbosity = 3
np.random.seed(26)

indir          = '/path/to/integrated/processed_data/'
csvdir         = '../data/processed/'
figdir         = '../figures/'
genelist_path  = csvdir + 'ext_fig3d_i_marker_lists.csv'

os.makedirs(csvdir, exist_ok=True)
os.makedirs(figdir, exist_ok=True)

plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams['font.family'] = 'Helvetica'
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

patient_colors = {
    'Patient1': '#d9d9d9', 'Patient4': '#bdbdbd', 'Patient5': '#969696',
    'Patient2': '#636363', 'Patient3': '#252525',
}

# Gene-signature names and the CSV outputs each one maps to.
gene_list_columns = ['Cytotoxicity', 'Exhaustion', 'TIL dysfunction']
signature_csv_map = {
    'Cytotoxicity':    ('ext_fig3d_boxplot.csv', 'ext_fig3g_pointplot_data.csv', 'ext_fig3g_pointplot_stats.csv'),
    'Exhaustion':      ('ext_fig3e_boxplot.csv', 'ext_fig3h_pointplot_data.csv', 'ext_fig3h_pointplot_stats.csv'),
    'TIL dysfunction': ('ext_fig3f_boxplot.csv', 'ext_fig3i_pointplot_data.csv', 'ext_fig3i_pointplot_stats.csv'),
}

# T-cell subtype palette for the score boxplots: ColorBrewer YlGnBu sampled
# at 5 classes (`sns.color_palette('YlGnBu', 5)`). Earlier and current label
# spellings share the same colour.
palette_map = {
    'Cytotoxic T':        '#ffffcc', 'Cytotoxic CD8T':     '#ffffcc',
    'Naive/CM T':         '#a1dab4',
    'Proliferating T':    '#41b6c4', 'Proliferating CD8T': '#41b6c4',
    'Treg':               '#2c7fb8',
    'γδT':                '#253494', 'gdT':                '#253494',
}

# Main-celltype palette for the stacked-bar neighbour-composition figure.
color_dict = {
    'Endothelial': '#d73027', 'Fibroblast': '#f46d43', 'Schwann': '#fdae61',
    'Neuroblast':  '#fee090', 'Macrophage': '#e0f3f8',
    'B':           '#abd9e9', 'T':          '#74add1',
    'Other':       '#E6E6E6',
}

## Helpers

In [ ]:
def get_neighbor_composition(adata, cell_mask, label_col='celltype'):
    connectivities = adata.obsp['spatial_connectivities']
    target_indices = np.where(cell_mask)[0]
    if len(target_indices) == 0:
        return None
    all_neighbors = []
    labels = adata.obs[label_col].values
    for idx in target_indices:
        all_neighbors.extend(labels[connectivities[idx].indices])
    if not all_neighbors:
        return None
    return pd.Series(all_neighbors).value_counts(normalize=True)

def make_df_plot(sample_results, target_subtypes, top_n=10):
    if not sample_results:
        return None
    df_res = pd.DataFrame(sample_results).fillna(0)
    top_types = df_res.mean(axis=1).nlargest(top_n).index.tolist()
    df_top = df_res.loc[top_types].copy()
    other_row = (1.0 - df_top.sum(axis=0)).clip(lower=0)
    df_top.loc['Other'] = other_row
    return df_top.T.reindex(target_subtypes).fillna(0)

## Load h5ad + gene-list CSV

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')
genelist_df = pd.read_csv(genelist_path)
print(genelist_df.head())
gene_list_columns = ['Cytotoxicity', 'Exhaustion', 'TIL dysfunction']

## Score T cells and tabulate per-signature DX vs PT changes

For each gene signature, compute a per-cell score with `sc.tl.score_genes`,
record per-cell scores stratified by T-cell subtype, the per-patient mean
score at each timepoint, and the paired t-test / Wilcoxon test of DX vs PT.

In [ ]:
signature_csv_map = {
    'Cytotoxicity':    ('ext_fig3d_boxplot.csv', 'ext_fig3g_pointplot_data.csv', 'ext_fig3g_pointplot_stats.csv'),
    'Exhaustion':      ('ext_fig3e_boxplot.csv', 'ext_fig3h_pointplot_data.csv', 'ext_fig3h_pointplot_stats.csv'),
    'TIL dysfunction': ('ext_fig3f_boxplot.csv', 'ext_fig3i_pointplot_data.csv', 'ext_fig3i_pointplot_stats.csv'),
}

t_cells = adata[adata.obs['celltype'] == 'T'].copy()

for sig_name in gene_list_columns:
    boxplot_csv, pp_data_csv, pp_stats_csv = signature_csv_map[sig_name]
    score_name = sig_name.replace(' ', '_') + '_score'

    # Per-cell score using the signature gene list.
    genes = [str(g).strip() for g in genelist_df[sig_name].dropna().tolist() if str(g).strip()]
    sc.tl.score_genes(t_cells, gene_list=genes, score_name=score_name)

    # Per-cell scores stratified by T-cell subtype.
    box_df = (
        t_cells.obs[['T_subtype', score_name]]
        .rename(columns={score_name: f'{sig_name} score'})
        .dropna(subset=['T_subtype', f'{sig_name} score'])
        .copy()
    )
    box_df.to_csv(csvdir + boxplot_csv, index=False)

    # Per-patient mean score at DX and PT.
    data = []
    for s in t_cells.obs['sample'].unique():
        sub = t_cells[(t_cells.obs['sample'] == s) & (t_cells.obs['celltype'] == 'T')]
        if sub.n_obs < 10:
            continue
        data.append({
            'Patient':   sub.obs['patient'].iloc[0],
            'Timepoint': sub.obs['timepoint'].iloc[0],
            'Score':     sub.obs[score_name].mean(),
        })
    df_score = pd.DataFrame(data)
    paired = (
        df_score.pivot_table(index='Patient', columns='Timepoint', values='Score', aggfunc='mean')
        .dropna(subset=['DX', 'PT'])
    )
    paired.reset_index().to_csv(csvdir + pp_data_csv, index=False)

    # Paired t-test and Wilcoxon test of DX vs PT.
    diff = paired['PT'] - paired['DX']
    delta = diff.mean()
    p_val = ttest_rel(paired['PT'], paired['DX'], nan_policy='omit').pvalue
    try:
        p_val_w = wilcoxon(paired['PT'], paired['DX'], alternative='two-sided').pvalue
    except ValueError:
        p_val_w = np.nan
    summary_df = pd.DataFrame({
        'metric': ['n_pairs', 'mean_PT_minus_DX_delta', 'paired_ttest_p', 'wilcoxon_p'],
        'value':  [len(paired), float(delta),
                   float(p_val) if pd.notna(p_val) else np.nan,
                   float(p_val_w) if pd.notna(p_val_w) else np.nan],
    })
    summary_df.to_csv(csvdir + pp_stats_csv, index=False)
    print(f'\n--- {sig_name}: paired_t p={p_val:.3g}, wilcoxon p={p_val_w}, delta={delta:.4f}, n={len(paired)} ---')

## Per-signature boxplot of scores by T-cell subtype (Ext Fig 3 d/e/f)

In [ ]:
for sig_name in gene_list_columns:
    boxplot_csv, _, _ = signature_csv_map[sig_name]
    score_col = f'{sig_name} score'
    box_df = pd.read_csv(csvdir + boxplot_csv)
    medians = box_df.groupby('T_subtype')[score_col].median().sort_values(ascending=False)
    order = medians.index.tolist()

    fig, ax = plt.subplots(figsize=(4/2.54, 5/2.54))
    sns.boxplot(data=box_df, x='T_subtype', y=score_col, palette=palette_map,
                showfliers=False, order=order, ax=ax, linewidth=0.5,
                boxprops={'linewidth': 0.5}, whiskerprops={'linewidth': 0.5},
                capprops={'linewidth': 0.5}, medianprops={'linewidth': 0.5})
    ax.set_ylabel('Score')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right')
    ax.set_frame_on(False); ax.grid(False)
    plt.tight_layout()
    plt.savefig(figdir + f'boxplot_{sig_name.replace(" ", "_")}_T_subtypes.pdf',
                format='pdf', transparent=True, bbox_inches='tight')
    plt.show(); plt.close(fig)

## Per-patient DX vs PT mean score per signature (Ext Fig 3 g/h/i)

In [ ]:
for sig_name in gene_list_columns:
    _, pp_data_csv, pp_stats_csv = signature_csv_map[sig_name]
    paired = pd.read_csv(csvdir + pp_data_csv).set_index('Patient')
    stats  = pd.read_csv(csvdir + pp_stats_csv).set_index('metric')['value']

    plot_df = paired.reset_index().melt(id_vars='Patient', value_vars=['DX', 'PT'],
                                         var_name='Timepoint', value_name='Score')
    plot_df['Timepoint'] = pd.Categorical(plot_df['Timepoint'], categories=['DX', 'PT'], ordered=True)
    plot_df = plot_df.sort_values(['Patient', 'Timepoint'])

    fig, ax = plt.subplots(figsize=(4/2.54, 5/2.54))
    for patient in plot_df['Patient'].unique():
        pdata = plot_df[plot_df['Patient'] == patient]
        ax.plot(pdata['Timepoint'], pdata['Score'], 'o-',
                color=patient_colors.get(patient, 'gray'),
                linewidth=1.5, markersize=4)
    ax.set_ylabel('Mean score')
    ax.set_frame_on(False); ax.grid(False)

    def _fmt(x):
        return f'{x:.2g}' if pd.notna(x) else 'NA'
    n   = int(stats.get('n_pairs', np.nan)) if pd.notna(stats.get('n_pairs', np.nan)) else 0
    p_t = stats.get('paired_ttest_p', np.nan)
    p_w = stats.get('wilcoxon_p', np.nan)
    d   = stats.get('mean_PT_minus_DX_delta', np.nan)
    ax.text(0.5, 1.25, f'n={n}\npaired t p={_fmt(p_t)}\nwilcoxon p={_fmt(p_w)}\nΔ={_fmt(d)}',
            transform=ax.transAxes, verticalalignment='top', horizontalalignment='center')
    plt.tight_layout()
    plt.savefig(figdir + f'pointplot_{sig_name.replace(" ", "_")}_timepoints.pdf',
                format='pdf', transparent=True, bbox_inches='tight')
    plt.show(); plt.close(fig)

## Benjamini-Hochberg FDR across the three signatures

In [ ]:
files = ['ext_fig3g_pointplot_stats.csv', 'ext_fig3h_pointplot_stats.csv', 'ext_fig3i_pointplot_stats.csv']

dfs = []
for fp in files:
    df = pd.read_csv(csvdir + fp)
    df['source_file'] = fp
    dfs.append(df)
all_stats = pd.concat(dfs, ignore_index=True)
all_stats['value'] = pd.to_numeric(all_stats['value'], errors='coerce')

def _bh_fdr(s):
    """BH-FDR over a Series; preserves NaN positions."""
    out = pd.Series(np.nan, index=s.index, dtype=float)
    valid = s.notna()
    if valid.any():
        _, q = fdrcorrection(s[valid].values, alpha=0.05)
        out.loc[valid] = q
    return out

t = all_stats.loc[all_stats['metric'] == 'paired_ttest_p', ['source_file', 'value']].copy()
t = t.rename(columns={'value': 'paired_ttest_p'})
t['paired_ttest_p_fdr'] = _bh_fdr(t['paired_ttest_p'])

w = all_stats.loc[all_stats['metric'] == 'wilcoxon_p', ['source_file', 'value']].copy()
w = w.rename(columns={'value': 'wilcoxon_p'})
w['wilcoxon_p_fdr'] = _bh_fdr(w['wilcoxon_p'])

fdr_table = t.merge(w, on='source_file', how='outer')

score_lookup = {
    'ext_fig3g_pointplot_stats.csv': 'Cytotoxicity score',
    'ext_fig3h_pointplot_stats.csv': 'Exhaustion score',
    'ext_fig3i_pointplot_stats.csv': 'TIL dysfunction score',
}
fdr_table['score'] = fdr_table['source_file'].map(score_lookup)
fdr_table = fdr_table[['score', 'paired_ttest_p', 'paired_ttest_p_fdr', 'wilcoxon_p', 'wilcoxon_p_fdr']]
fdr_table.to_csv(csvdir + 'ext_fig3ghi_fdr.csv', index=False)
display(fdr_table)

## Top-10 spatial neighbors of each T-cell subtype (Fig 4a)

For each sample, build a spatial graph and, for each T-cell subtype, tally
the proportion of immediate neighbors falling into each main cell type.
The 10 most frequent neighbors per subtype are reported individually and
as a sample-weighted consensus.

In [ ]:
samples_to_plot = list(adata.obs['sample'].unique())
target_subtypes = ['γδT', 'Treg', 'Naive/CM T', 'Proliferating T', 'Cytotoxic T']
weighting = 'equal'

sample_to_results = {}
results_by_subtype = {st: [] for st in target_subtypes}
weights_by_subtype = {st: [] for st in target_subtypes}

for sample in samples_to_plot:
    print(f'Neighbor composition: {sample}')
    adata_sub = adata[adata.obs['sample'] == sample].copy()
    if adata_sub.n_obs < 10:
        continue
    sq.gr.spatial_neighbors(adata_sub, coord_type='generic', spatial_key='spatial', n_neighs=10)
    sample_results = {}
    for subtype in target_subtypes:
        is_subtype = adata_sub.obs['T_subtype'] == subtype
        if int(is_subtype.sum()) < 10:
            continue
        res = get_neighbor_composition(adata_sub, is_subtype, label_col='celltype')
        if res is None:
            continue
        sample_results[subtype] = res
        results_by_subtype[subtype].append(res)
        weights_by_subtype[subtype].append(1.0 if weighting == 'equal' else float(is_subtype.sum()))
    if sample_results:
        sample_to_results[sample] = sample_results

source_rows = []
top_n = 10
for sample, sample_results in sample_to_results.items():
    df_plot = make_df_plot(sample_results, target_subtypes, top_n=top_n)
    if df_plot is None:
        continue
    df_plot = df_plot.reindex(target_subtypes).fillna(0)
    long_df = (df_plot.reset_index(names='T_subtype')
               .melt(id_vars='T_subtype', var_name='Neighbor_CellType', value_name='Proportion'))
    long_df['Plot_Type'] = 'individual'
    long_df['Plot_ID']   = sample
    long_df['Top_N']     = top_n
    long_df['Weighting'] = weighting
    source_rows.append(long_df)

consensus_results = {}
for subtype, series_list in results_by_subtype.items():
    if not series_list:
        continue
    df = pd.concat(series_list, axis=1).fillna(0)
    w = np.array(weights_by_subtype[subtype], dtype=float)
    consensus_results[subtype] = (df * w).sum(axis=1) / w.sum()

if consensus_results:
    df_plot_cons = make_df_plot(consensus_results, target_subtypes, top_n=top_n).reindex(target_subtypes).fillna(0)
    long_cons = (df_plot_cons.reset_index(names='T_subtype')
                 .melt(id_vars='T_subtype', var_name='Neighbor_CellType', value_name='Proportion'))
    long_cons['Plot_Type'] = 'consensus'
    long_cons['Plot_ID']   = 'consensus'
    long_cons['Top_N']     = top_n
    long_cons['Weighting'] = weighting
    source_rows.append(long_cons)

source_data = pd.concat(source_rows, ignore_index=True)
source_data = source_data[source_data['Neighbor_CellType'] != 'Other'].copy()
source_data = source_data[['Plot_Type', 'Plot_ID', 'T_subtype', 'Neighbor_CellType', 'Proportion', 'Top_N', 'Weighting']]

source_data_wide = source_data.pivot_table(
    index=['Plot_Type', 'Plot_ID', 'T_subtype', 'Top_N', 'Weighting'],
    columns='Neighbor_CellType', values='Proportion', aggfunc='first', fill_value=0,
).reset_index()

# Extended fig 4a: all individuals + consensus
source_data_wide.to_csv(csvdir + 'ext_fig4a.csv', index=False)
# Main fig 4a: consensus rows only
source_data_wide[source_data_wide['Plot_Type'] == 'consensus'].to_csv(csvdir + 'fig4a.csv', index=False)

In [ ]:
# Stacked-bar neighbour composition per sample and consensus.
from matplotlib.patches import Patch

source_data_wide = pd.read_csv(csvdir + 'ext_fig4a.csv')

# Align T-cell subtype labels to the Unicode forms.
source_data_wide['T_subtype'] = source_data_wide['T_subtype'].replace({'gdT': 'γδT'})

neighbor_cols = [c for c in source_data_wide.columns
                 if c not in ('Plot_Type', 'Plot_ID', 'T_subtype', 'Top_N', 'Weighting')]
target_subtypes_local = ['γδT', 'Treg', 'Naive/CM T', 'Proliferating T', 'Cytotoxic T']

def plot_neighbor_composition(df_plot, title, outpath, color_dict, x_categories,
                              show_legend=True, figsize=(5/2.54, 8/2.54)):
    df_plot = df_plot.reindex(x_categories).fillna(0)
    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(x_categories))
    for i, subtype in enumerate(x_categories):
        row = df_plot.loc[subtype]
        bottom = 0.0
        for col in df_plot.columns:
            val = row[col]
            if val <= 0:
                continue
            ax.bar(x[i], val, bottom=bottom, color=color_dict.get(col, '#E6E6E6'),
                   edgecolor='white', linewidth=0.5, width=0.95)
            if val >= 0.05:
                ax.text(x[i], bottom + val / 2, f'{val:.2f}', ha='center', va='center')
            bottom += val
    ax.set_title(title); ax.set_xlabel('T Cell Subtype'); ax.set_ylabel('Proportion of Neighbors')
    ax.set_xticks(x); ax.set_xticklabels(x_categories, rotation=90, ha='right')
    ax.set_xlim(-0.5, len(x_categories) - 0.5); ax.set_ylim(0, 1.0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', length=0)
    ax.grid(False)
    if show_legend:
        legend_handles = [Patch(facecolor=color_dict.get(c, '#E6E6E6'), label=c) for c in df_plot.columns]
        ax.legend(handles=legend_handles, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    plt.tight_layout()
    plt.savefig(outpath, format='pdf', transparent=True)
    plt.show(); plt.close(fig)

for plot_id, df_g in source_data_wide.groupby('Plot_ID'):
    df_plot = df_g.set_index('T_subtype')[neighbor_cols]
    plot_neighbor_composition(
        df_plot=df_plot, title=f'Top10 Neighbor by T: {plot_id}',
        outpath=figdir + f'top10neighbor_T_{plot_id}.pdf',
        color_dict=color_dict, x_categories=target_subtypes_local, show_legend=False,
    )